# 10 — Model Explainability with SHAP

**Objective:** Compute global and local feature attributions using SHAP (SHapley Additive exPlanations) to explain churn model decision making.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import shap
from pathlib import Path
from typing import Dict, Any, Optional

from src.config import *
from src.utils import logger, timer, load_model

c:\Users\admin\OneDrive\Desktop\airline-loyality\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Feature Name Mapping

Define technical to business name mappings for customer features.

In [2]:
FEATURE_BUSINESS_NAMES = {
    "Days_Since_Last_Flight": "Days since last flight",
    "Days_Since_Last_Earning": "Days since last points earned",
    "Days_Since_Last_Redemption": "Days since last points redeemed",
    "Flights_3M": "Flights in last 3 months",
    "Flights_6M": "Flights in last 6 months",
    "Flights_12M": "Flights in last 12 months",
    "Flights_Lifetime": "Total lifetime flights",
    "Active_Months": "Number of active months",
    "Points_Earned_3M": "Points earned (3 months)",
    "Points_Earned_6M": "Points earned (6 months)",
    "Points_Earned_12M": "Points earned (12 months)",
    "Points_Redeemed_3M": "Points redeemed (3 months)",
    "Points_Redeemed_6M": "Points redeemed (6 months)",
    "Points_Redeemed_12M": "Points redeemed (12 months)",
    "Distance_3M": "Distance flown (3 months)",
    "Distance_6M": "Distance flown (6 months)",
    "Distance_12M": "Distance flown (12 months)",
    "Points_Earned_Lifetime": "Total lifetime points earned",
    "Points_Redeemed_Lifetime": "Total lifetime points redeemed",
    "Distance_Lifetime": "Total lifetime distance",
    "Dollar_Cost_Lifetime": "Lifetime dollar cost of redeemed points",
    "Flight_Trend_6M": "Flight activity trend (6 months)",
    "Points_Trend_6M": "Points earning trend (6 months)",
    "Distance_Trend_6M": "Distance trend (6 months)",
    "Engagement_Trend": "Overall engagement trend",
    "Loyalty_Card_Ordinal": "Loyalty tier level",
    "Tenure_Months": "Membership tenure (months)",
    "Education_Ordinal": "Education level",
    "Salary": "Annual salary",
    "CLV": "Customer lifetime value (CLV)",
    "Salary_Missing": "Salary was missing (imputed)",
    "Enrollment_Age_Months": "Months since enrollment",
    "Is_Promo_Enrollment": "Enrolled via promotion",
    "Is_Male": "Gender (Male)",
    "Is_Married": "Married",
    "Is_Single": "Single",
    "Redemption_Ratio": "Points redemption ratio",
    "Avg_Distance_Per_Flight": "Average distance per flight",
    "Activity_Consistency": "Activity consistency score",
    "Preferred_Quarter": "Preferred travel quarter",
    "Declining_Activity": "Declining flight activity",
    "Declining_Redemption": "Declining redemption activity",
    "Months_Inactive": "Consecutive months inactive",
}

def get_business_name(feature: str) -> str:
    return FEATURE_BUSINESS_NAMES.get(feature, feature)

## 2. SHAP Computation & Global Analysis

Initialize SHAP TreeExplainer (falling back to KernelExplainer if necessary) and summarize mean absolute attribution values to identify top global churn drivers.

In [3]:
def compute_shap_values(
    model, X: pd.DataFrame, max_samples: int = 1000
) -> shap.Explanation:
    if len(X) > max_samples:
        X_sample = X.sample(max_samples, random_state=RANDOM_SEED)
    else:
        X_sample = X.copy()

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer(X_sample)
        logger.info(f"SHAP values computed with TreeExplainer for {len(X_sample)} samples")
    except Exception:
        explainer = shap.KernelExplainer(model.predict_proba, X_sample.iloc[:100])
        shap_values = explainer(X_sample)
        logger.info(f"SHAP values computed with KernelExplainer for {len(X_sample)} samples")

    return shap_values

def global_shap_analysis(
    model, X: pd.DataFrame, max_samples: int = 1000
) -> Dict[str, Any]:
    shap_values = compute_shap_values(model, X, max_samples)

    if len(shap_values.shape) == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values

    mean_abs_shap = np.abs(shap_vals.values).mean(axis=0)
    feature_importance = pd.DataFrame({
        "Feature": X.columns,
        "Mean_SHAP": mean_abs_shap,
        "Business_Name": [get_business_name(f) for f in X.columns],
    }).sort_values("Mean_SHAP", ascending=False)

    top_drivers = feature_importance.head(10)
    logger.info(f"\nTop 10 Churn Drivers:")
    for _, row in top_drivers.iterrows():
        logger.info(f"  {row['Business_Name']}: SHAP = {row['Mean_SHAP']:.4f}")

    return {
        "shap_values": shap_vals,
        "feature_importance": feature_importance,
        "top_drivers": top_drivers,
        "X_sample": X.iloc[:max_samples] if len(X) > max_samples else X,
    }

## 3. Local Customer Diagnostics

Isolate SHAP values for individual customers to understand the specific behaviors driving their risk score.

In [4]:
def local_shap_explanation(
    model, X: pd.DataFrame, customer_idx: int, shap_values=None
) -> Dict[str, Any]:
    if shap_values is None:
        shap_values = compute_shap_values(model, X.iloc[[customer_idx]])

    if len(shap_values.shape) == 3:
        vals = shap_values[:, :, 1]
    else:
        vals = shap_values

    if hasattr(vals, 'values'):
        customer_shap = vals.values[0] if len(vals.values.shape) > 1 else vals.values
    else:
        customer_shap = vals[0]

    explanation = pd.DataFrame({
        "Feature": X.columns,
        "Value": X.iloc[customer_idx].values,
        "SHAP_Value": customer_shap,
        "Business_Name": [get_business_name(f) for f in X.columns],
        "Direction": ["↑ Increases churn risk" if s > 0 else "↓ Decreases churn risk"
                      for s in customer_shap],
    }).sort_values("SHAP_Value", key=abs, ascending=False)

    return {
        "explanation": explanation,
        "top_risk_factors": explanation[explanation["SHAP_Value"] > 0].head(5),
        "top_retention_factors": explanation[explanation["SHAP_Value"] < 0].head(5),
    }

def generate_customer_explanation_text(explanation: Dict[str, Any]) -> str:
    risk_factors = explanation["top_risk_factors"]
    retention_factors = explanation["top_retention_factors"]

    text = "**Why this customer may churn:**\n"
    for _, row in risk_factors.iterrows():
        text += f"  • {row['Business_Name']} = {row['Value']:.1f}\n"

    text += "\n**Factors keeping this customer:**\n"
    for _, row in retention_factors.iterrows():
        text += f"  • {row['Business_Name']} = {row['Value']:.1f}\n"

    return text

## 4. Run Explainability Diagnostic

Execute global SHAP analysis and print top global predictors of customer churn.

In [5]:
model = load_model(MODELS_DIR / "churn_model.joblib")
df = pd.read_csv(FEATURES_DIR / "customer_features.csv")
feature_cols = [c for c in df.columns if c not in [PK, "Churn"]]
X = df[feature_cols]

global_importance = global_shap_analysis(model, X, max_samples=250)
print(global_importance["top_drivers"])

2026-06-12 11:29:52 | airline_loyalty | INFO | Loading model: churn_model.joblib
2026-06-12 11:29:52 | airline_loyalty | INFO | SHAP values computed with TreeExplainer for 250 samples
2026-06-12 11:29:52 | airline_loyalty | INFO | 
Top 10 Churn Drivers:
2026-06-12 11:29:52 | airline_loyalty | INFO |   Membership tenure (months): SHAP = 2.2485
2026-06-12 11:29:52 | airline_loyalty | INFO |   Months since enrollment: SHAP = 2.1214
2026-06-12 11:29:52 | airline_loyalty | INFO |   Days since last flight: SHAP = 1.4319
2026-06-12 11:29:52 | airline_loyalty | INFO |   Activity consistency score: SHAP = 0.2851
2026-06-12 11:29:52 | airline_loyalty | INFO |   Points redeemed (3 months): SHAP = 0.1337
2026-06-12 11:29:52 | airline_loyalty | INFO |   Days since last points redeemed: SHAP = 0.1305
2026-06-12 11:29:52 | airline_loyalty | INFO |   Customer lifetime value (CLV): SHAP = 0.1240
2026-06-12 11:29:52 | airline_loyalty | INFO |   Overall engagement trend: SHAP = 0.1071
2026-06-12 11:29:52

                       Feature  Mean_SHAP                    Business_Name
26               Tenure_Months   2.248539       Membership tenure (months)
31       Enrollment_Age_Months   2.121432          Months since enrollment
0       Days_Since_Last_Flight   1.431940           Days since last flight
38        Activity_Consistency   0.285120       Activity consistency score
9           Points_Redeemed_3M   0.133653       Points redeemed (3 months)
2   Days_Since_Last_Redemption   0.130507  Days since last points redeemed
29                         CLV   0.123984    Customer lifetime value (CLV)
24            Engagement_Trend   0.107101         Overall engagement trend
28                      Salary   0.104845                    Annual salary
37     Avg_Distance_Per_Flight   0.094359      Average distance per flight
